# Deploy the Reference H2O Endpoint

Generate the scoring function and pinned environment, register the reference bundle, deploy at zero traffic, verify golden parity, and optionally promote traffic.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/04_deploy_managed_online_endpoint.ipynb` and the Azure ML managed endpoint examples.

In [ ]:
from pathlib import Path
import ast
import json
import os
import textwrap
import yaml

import numpy as np
import pandas as pd
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes, ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    Environment,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    Model,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
bundle_value = Path(os.environ["H2O_BUNDLE_DIR"])
BUNDLE_DIR = bundle_value if bundle_value.is_absolute() else WORKSHOP_ROOT / bundle_value
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
ENVIRONMENT_NAME = os.environ["H2O_ENVIRONMENT_NAME"]
ENDPOINT_NAME = os.environ["H2O_ENDPOINT_NAME"]
DEPLOYMENT_NAME = os.environ["H2O_DEPLOYMENT_NAME"]
IDENTITY_ID = os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip()
DEPLOY = os.getenv("DEPLOY_H2O_ENDPOINT", "false").lower() in {"1", "true", "yes"}
PROMOTE = os.getenv("PROMOTE_H2O_TRAFFIC", "false").lower() in {"1", "true", "yes"}
manifest = json.loads((BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8"))

generated_dir = WORKSHOP_ROOT / "outputs/generated/h2o_reference/online"
generated_code_dir = generated_dir / "code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
score_path = generated_code_dir / "score.py"
conda_path = generated_dir / "conda.yaml"
conda_source = textwrap.dedent(f"""
name: h2o-reference-online
channels:
  - conda-forge
dependencies:
  - python=3.12
  - openjdk=17
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - h2o=={manifest['h2o_version']}
      - numpy==1.26.4
      - pandas==2.2.3
""").lstrip()
yaml.safe_load(conda_source)
conda_path.write_text(conda_source, encoding="utf-8")

score_source = r'''
import atexit
import hashlib
import json
import logging
import os
import threading
from pathlib import Path

import h2o
import pandas as pd
from azureml_inference_server_http.api.aml_response import AMLResponse

_model = None
_manifest = None
_lock = threading.Lock()


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _shutdown():
    try:
        if h2o.connection() is not None:
            h2o.cluster().shutdown(prompt=False)
    except Exception:
        logging.exception("H2O shutdown failed")


def init():
    global _model, _manifest
    root = Path(os.environ["AZUREML_MODEL_DIR"])
    matches = list(root.rglob("model_manifest.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one manifest, found {len(matches)}")
    manifest_path = matches[0]
    _manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if _manifest.get("model_format") != "h2o_binary":
        raise RuntimeError("Reference asset must be a native H2O binary")
    if h2o.__version__ != _manifest["h2o_version"]:
        raise RuntimeError(f"Expected h2o=={_manifest['h2o_version']}, found {h2o.__version__}")
    model_path = manifest_path.parent / _manifest["model_file"]
    if _sha256(model_path) != _manifest["files"][model_path.name]:
        raise RuntimeError("Model checksum does not match the manifest")
    h2o.no_progress()
    h2o.init(
        ip="127.0.0.1", port=54321, start_h2o=True,
        nthreads=int(os.environ.get("H2O_NTHREADS", "3")),
        max_mem_size=os.environ.get("H2O_MAX_MEM_SIZE", "6G"),
        strict_version_check=True, bind_to_localhost=True,
        verbose=False, telemetry=False,
    )
    _model = h2o.load_model(str(model_path))
    atexit.register(_shutdown)


def run(raw_data):
    try:
        payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
        input_data = payload.get("input_data", {})
        columns = input_data.get("columns")
        rows = input_data.get("data")
        if columns != _manifest["features"]:
            raise ValueError(f"Expected columns in this order: {_manifest['features']}")
        if not isinstance(rows, list) or not rows:
            raise ValueError("Request must contain at least one row")
        frame = pd.DataFrame(rows, columns=columns)
        categorical = set(_manifest.get("categorical_features", []))
        for column in set(columns) - categorical:
            frame[column] = pd.to_numeric(frame[column], errors="raise")
        for column in categorical:
            frame[column] = frame[column].astype("string")
    except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
        return AMLResponse({"error": str(exc)}, 400, json_str=True)
    h2o_frame = h2o.H2OFrame(frame)
    prediction_frame = None
    try:
        for column in _manifest.get("categorical_features", []):
            h2o_frame[column] = h2o_frame[column].asfactor()
        with _lock:
            prediction_frame = _model.predict(h2o_frame)
            values = prediction_frame.as_data_frame()["predict"]
        predictions = [value.item() if hasattr(value, "item") else value for value in values]
    finally:
        if prediction_frame is not None:
            h2o.remove(prediction_frame)
        h2o.remove(h2o_frame)
    return {
        "predictions": predictions,
        "model_name": _manifest["model_name"],
        "model_version": _manifest["model_version"],
    }
'''
score_source = textwrap.dedent(score_source).lstrip()
ast.parse(score_source, filename=str(score_path))
score_path.write_text(score_source, encoding="utf-8")
print(f"Generated scoring script: {score_path}")
print(f"Generated Conda file: {conda_path}")

model_definition = Model(
    name=MODEL_NAME,
    type=AssetTypes.CUSTOM_MODEL,
    path=str(BUNDLE_DIR),
    description="Validated workshop H2O binary-model bundle",
    tags={"workshop": "azureml-h2o", "model_format": "h2o_binary", "h2o_version": manifest["h2o_version"]},
)
environment_definition = Environment(
    name=ENVIRONMENT_NAME,
    image="mcr.microsoft.com/azureml/minimal-py312-inference:latest",
    conda_file=str(conda_path),
    description="OpenJDK 17 and exact H2O runtime for online scoring",
    tags={"workshop": "azureml-h2o", "h2o_version": manifest["h2o_version"]},
)
identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[ManagedIdentityConfiguration(resource_id=IDENTITY_ID)],
    )
endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    auth_mode="aad_token",
    identity=identity,
    public_network_access=os.getenv("AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS", "disabled"),
    description="Workshop H2O binary-model endpoint",
    tags={"workshop": "azureml-h2o"},
)
if DEPLOY:
    registered_model = ml_client.models.create_or_update(model_definition)
    verified_model = ml_client.models.get(
        MODEL_NAME, version=registered_model.version
    )
    print(f"Registered model: {verified_model.name}:{verified_model.version}")

    registered_environment = ml_client.environments.create_or_update(environment_definition)
    verified_environment = ml_client.environments.get(
        ENVIRONMENT_NAME, version=registered_environment.version
    )
    print(
        f"Registered environment: "
        f"{verified_environment.name}:{verified_environment.version}"
    )
    deployment_definition = ManagedOnlineDeployment(
        name=DEPLOYMENT_NAME,
        endpoint_name=ENDPOINT_NAME,
        model=verified_model,
        environment=verified_environment,
        code_configuration=CodeConfiguration(
            code=str(generated_code_dir), scoring_script="score.py"
        ),
        instance_type=os.environ["AZUREML_ONLINE_INSTANCE_TYPE"],
        instance_count=1,
        app_insights_enabled=True,
        environment_variables={
            "WORKER_COUNT": "1",
            "H2O_NTHREADS": os.environ["H2O_NTHREADS"],
            "H2O_MAX_MEM_SIZE": os.environ["H2O_MAX_MEM_SIZE"],
        },
    )
    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
    except ResourceNotFoundError:
        endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
    deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
    if deployment.provisioning_state != "Succeeded":
        raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")
    logs = ml_client.online_deployments.get_logs(DEPLOYMENT_NAME, ENDPOINT_NAME, 100, container_type="inference-server")
    print("\n".join(logs.splitlines()[-25:]))

    golden_input = pd.read_csv(BUNDLE_DIR / "golden_input.csv")
    golden_expected = pd.read_csv(BUNDLE_DIR / "golden_expected.csv")
    request = {"input_data": {"columns": manifest["features"], "data": golden_input[manifest["features"]].values.tolist()}}
    request_path = generated_dir / "request.json"
    request_path.write_text(json.dumps(request, indent=2), encoding="utf-8")
    response = json.loads(ml_client.online_endpoints.invoke(endpoint_name=ENDPOINT_NAME, deployment_name=DEPLOYMENT_NAME, request_file=str(request_path)))
    np.testing.assert_allclose(golden_expected["predict"], response["predictions"], rtol=1e-6, atol=1e-6)
    print(f"Cloud parity passed for {len(response['predictions'])} rows.")

    if PROMOTE:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        endpoint.traffic = {DEPLOYMENT_NAME: 100}
        ml_client.online_endpoints.begin_create_or_update(endpoint).result()
        print(f"Traffic promoted to {DEPLOYMENT_NAME}")
    else:
        print("Traffic remains unchanged. Set PROMOTE_H2O_TRAFFIC=true to promote.")
else:
    print(f"Prepared {ENDPOINT_NAME}/{DEPLOYMENT_NAME}; set DEPLOY_H2O_ENDPOINT=true to deploy.")

## Expected Result

The notebook-generated environment and scorer deploy successfully, direct invocation matches golden predictions, and traffic changes only when enabled.

Next: `05_submit_reference_scoring_pipeline.ipynb`.